In [1]:
# ==========================================
# CELDA 1: Configuración y Carga de Infraestructura
# ==========================================
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from ortools.sat.python import cp_model
import warnings
warnings.filterwarnings('ignore')

# 1. Conectar a la BD
user = "Joasro"
password = "Akriila123." 
host = "localhost"
db = "dss_academico_unah"

engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{db}")

print("Cargando logística desde la BD...")

# Traer tablas logísticas
df_docentes = pd.read_sql("SELECT * FROM docentes_activos", engine)
df_docente_area = pd.read_sql("SELECT * FROM docente_area", engine)
df_aulas = pd.read_sql("SELECT * FROM espacios_fisicos", engine)

# Malla con las columnas nuevas de modalidad y laboratorios
df_malla = pd.read_sql("""
    SELECT ID_Clase, Nombre_Clase, Codigo_Oficial, ID_Area, 
           Requiere_Laboratorio, Permite_Presencial, Permite_Virtual 
    FROM malla_curricular
""", engine)

# 2. Cargar la demanda proyectada por la IA
df_demanda = pd.read_csv('../data/demanda_proyectada_2026.csv')

# 🔥 EL FIX: Recuperamos el ID_Clase cruzando los datos con la malla curricular
if 'ID_Clase' not in df_demanda.columns:
    df_demanda = pd.merge(df_demanda, df_malla[['Codigo_Oficial', 'ID_Clase']], on='Codigo_Oficial', how='left')

# Parámetros Globales
HORARIOS = [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20] # Horas de clase
CAPACIDAD_ESTANDAR = 25 
ID_AULA_VIRTUAL = 999 

# SIMULADOR DE RESTRICCIONES HORARIAS: 
docentes_no_disponibles = {}

print(f"✅ Datos listos: {len(df_docentes)} docentes disponibles y {len(df_aulas)} aulas físicas.")

Cargando logística desde la BD...
✅ Datos listos: 13 docentes disponibles y 8 aulas físicas.


In [2]:
# ==========================================
# CELDA 2: Generador de Posibilidades (Con Límite de Turno Corregido)
# ==========================================
model = cp_model.CpModel()
secciones_posibles = {}

# Función robusta para sacar la hora de la BD sin que falle
def extraer_hora_fin(val):
    if pd.isnull(val): return 21
    try:
        if hasattr(val, 'components'): return val.components.hours # Si es timedelta
        if hasattr(val, 'hour'): return val.hour # Si es datetime.time
        if isinstance(val, str): return int(val.split(':')[0]) # Si es string '17:00'
    except: pass
    return 21 # Por defecto 9 PM si hay un error en el dato

print("Generando variables de decisión...")

for idx, row in df_demanda.iterrows():
    id_clase = row['ID_Clase']
    num_secciones = int(np.ceil(row['Cupos_Estimados'] / CAPACIDAD_ESTANDAR))
    
    info = df_malla[df_malla['ID_Clase'] == id_clase].iloc[0]
    p_presencial = info['Permite_Presencial']
    p_virtual = info['Permite_Virtual']
    area_clase = info['ID_Area']
    necesita_lab = info['Requiere_Laboratorio']

    for s in range(num_secciones):
        for d_idx, doc in df_docentes.iterrows():
            id_docente = doc['ID_Docente']
            tipo_d = doc['Tipo_Docente']
            
            # FILTRO 1: Especialidad del área
            especialidades = df_docente_area[df_docente_area['ID_Docente'] == id_docente]['ID_Area'].values
            if area_clase not in especialidades: continue

            # EXTRAER LA HORA REAL DE SALIDA
            hora_fin_turno = extraer_hora_fin(doc.get('Hora_Fin_Turno', None))

            for h in HORARIOS:
                # 🛑 FILTRO 2: LÍMITE DE TURNO CORREGIDO
                # Si la hora de clase es igual o mayor a su hora de salida, lo ignoramos.
                if h >= hora_fin_turno:
                    continue

                if id_docente in docentes_no_disponibles and h in docentes_no_disponibles[id_docente]:
                    continue

                # Regla de Tegucigalpa: Solo Virtual
                if tipo_d == 'Tegucigalpa':
                    if p_virtual == 1:
                        v_key = (id_clase, s, id_docente, h, ID_AULA_VIRTUAL, 'Virtual')
                        secciones_posibles[v_key] = model.NewBoolVar(f"c{id_clase}_s{s}_d{id_docente}_h{h}_TGU_VIRT")
                    continue

                # Opción Virtual (Base y Emergente)
                if p_virtual == 1:
                    v_key = (id_clase, s, id_docente, h, ID_AULA_VIRTUAL, 'Virtual')
                    secciones_posibles[v_key] = model.NewBoolVar(f"c{id_clase}_s{s}_d{id_docente}_h{h}_VIRT")
                
                # Opción Presencial (Base y Emergente)
                if p_presencial == 1:
                    for a_idx, aula in df_aulas.iterrows():
                        es_lab_aula = 1 if aula['Tipo_Espacio'].lower() == 'laboratorio' else 0
                        if necesita_lab != es_lab_aula: continue
                        
                        v_key = (id_clase, s, id_docente, h, aula['ID_Espacio'], 'Presencial')
                        secciones_posibles[v_key] = model.NewBoolVar(f"c{id_clase}_s{s}_d{id_docente}_h{h}_a{aula['ID_Espacio']}_PRES")

print(f"🧩 Se crearon {len(secciones_posibles)} combinaciones.")

Generando variables de decisión...
🧩 Se crearon 2994 combinaciones.


In [3]:
# ==========================================
# CELDA 3: Restricciones de Logística y Curriculares
# ==========================================

todas_las_variables = list(secciones_posibles.values())

# 1. Rango Curricular (21 a 28 secciones)
model.Add(sum(todas_las_variables) >= 21)
model.Add(sum(todas_las_variables) <= 28)

# 2. Choques de Aula (Evitar que 2 clases usen el mismo lab)
for id_aula in df_aulas['ID_Espacio']:
    for h in HORARIOS:
        clases_aula_hora = [var for (id_c, s, d, hora, a, mod), var in secciones_posibles.items() if a == id_aula and hora == h and mod != 'Virtual']
        if clases_aula_hora:
            model.Add(sum(clases_aula_hora) <= 1)

# 3. Choques de Docente y CARGA MÁXIMA
docentes_usados = {} # Para saber si el docente da al menos 1 clase

for id_doc in df_docentes['ID_Docente']:
    # Evitar choques de hora para el mismo profe
    for h in HORARIOS:
        clases_doc_hora = [var for (id_c, s, d, hora, a, mod), var in secciones_posibles.items() if d == id_doc and hora == h]
        if clases_doc_hora:
            model.Add(sum(clases_doc_hora) <= 1)
            
    # 🛑 CARGA MÁXIMA POR PROFESOR (Evita que los Base acaparen todo)
    todas_las_clases_docente = [var for (id_c, s, d, hora, a, mod), var in secciones_posibles.items() if d == id_doc]
    if todas_las_clases_docente:
        # Asignamos que un docente puede dar un máximo de 5 secciones por periodo
        model.Add(sum(todas_las_clases_docente) <= 5) 
        
        # Variable booleana para saber si "se contrató/usó" a este docente
        docentes_usados[id_doc] = model.NewBoolVar(f'docente_activo_{id_doc}')
        
        # Si la suma de clases es > 0, entonces docentes_usados[id_doc] = 1
        model.Add(sum(todas_las_clases_docente) > 0).OnlyEnforceIf(docentes_usados[id_doc])
        model.Add(sum(todas_las_clases_docente) == 0).OnlyEnforceIf(docentes_usados[id_doc].Not())

# 4. Una sección específica se programa MÁXIMO una vez
for idx, row in df_demanda.iterrows():
    id_c = row['ID_Clase']
    num_s = int(np.ceil(row['Cupos_Estimados'] / CAPACIDAD_ESTANDAR))
    for s in range(num_s):
        instancias_seccion = [var for (c, sec, d, h, a, mod), var in secciones_posibles.items() if c == id_c and sec == s]
        if instancias_seccion:
            model.Add(sum(instancias_seccion) <= 1)

# 🛑 5. LÍMITE DE CONTRATACIÓN (Máximo 2 Emergentes y 2 TGU)
emergentes_activos = [docentes_usados[doc['ID_Docente']] for idx, doc in df_docentes.iterrows() 
                      if doc['Tipo_Docente'] == 'Emergente' and doc['ID_Docente'] in docentes_usados]
tgu_activos = [docentes_usados[doc['ID_Docente']] for idx, doc in df_docentes.iterrows() 
               if doc['Tipo_Docente'] == 'Tegucigalpa' and doc['ID_Docente'] in docentes_usados]

if emergentes_activos:
    model.Add(sum(emergentes_activos) <= 2) # Máximo 2 ingenieros emergentes diferentes

if tgu_activos:
    model.Add(sum(tgu_activos) <= 2) # Máximo 2 ingenieros de Tegucigalpa diferentes

print("⚖️ Restricciones aplicadas: Carga máxima de docentes ajustada y límite de contrataciones activado.")

⚖️ Restricciones aplicadas: Carga máxima de docentes ajustada y límite de contrataciones activado.


In [4]:
# ==========================================
# CELDA 4: Función Objetivo (Prioridades de Jefatura)
# ==========================================

objetivo = []

for (id_c, s, id_d, h, id_a, mod), var in secciones_posibles.items():
    puntos = 0
    tipo_d = df_docentes[df_docentes['ID_Docente'] == id_d]['Tipo_Docente'].values[0]
    
    # --- ESTRATEGIA PRESUPUESTARIA ---
    if tipo_d == 'Base': 
        puntos += 20          # Prioridad máxima: aprovechar a los docentes fijos
    elif tipo_d == 'Tegucigalpa': 
        puntos += 5           # Segunda opción para cubrir demanda en formato virtual
    else: 
        puntos += 1           # Emergente (Último recurso, requiere nuevo presupuesto)
    
    # --- ESTRATEGIA DE HORARIOS ---
    if mod == 'Virtual':
        if h >= 17:
            puntos += 10      # Gran premio por virtual de noche (Tendencia real)
        elif h < 12:
            puntos -= 5       # Penalización por virtual en la mañana

    objetivo.append(var * puntos)

# Instrucción al algoritmo: Maximizar la puntuación
model.Maximize(sum(objetivo))

# EJECUTAR SOLVER
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print(f"✅ ¡Horario generado exitosamente! (Status: {solver.StatusName(status)})")
else:
    print("❌ No se pudo encontrar una solución. Verifica las restricciones.")

✅ ¡Horario generado exitosamente! (Status: OPTIMAL)


In [5]:
# ==========================================
# CELDA 5: Reporte Final y Sugerencias a Futuro
# ==========================================
horario_final = []
secciones_abiertas_por_clase = {}

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    # 1. Extraer las decisiones ganadoras
    for (id_c, s, id_d, h, id_a, mod), var in secciones_posibles.items():
        if solver.Value(var) == 1: 
            clase = df_malla[df_malla['ID_Clase'] == id_c]['Nombre_Clase'].values[0]
            docente = df_docentes[df_docentes['ID_Docente'] == id_d]['Nombre'].values[0]
            tipo_d = df_docentes[df_docentes['ID_Docente'] == id_d]['Tipo_Docente'].values[0]
            
            # Nombre del aula
            if mod == 'Virtual':
                aula_nombre = 'Campus Virtual'
            else:
                aula_nombre = df_aulas[df_aulas['ID_Espacio'] == id_a]['Nombre_Espacio'].values[0]
            
            horario_final.append({
                'Hora': f"{h}:00", 'Clase': clase, 'Seccion': f"Sec-{s+1}", 
                'Modalidad': mod, 'Docente': docente, 'Tipo_Contrato': tipo_d, 'Espacio': aula_nombre
            })
            
            # Contabilizar para el análisis de déficit
            secciones_abiertas_por_clase[id_c] = secciones_abiertas_por_clase.get(id_c, 0) + 1

    df_horario = pd.DataFrame(horario_final).sort_values(by=['Hora', 'Clase'])
    
    # ---------------------------------------------------------
    # TABLA 1: LA OFERTA ACADÉMICA
    # ---------------------------------------------------------
    print("🎓 OFERTA ACADÉMICA GENERADA (Optimizada para el próximo periodo):")
    pd.set_option('display.max_rows', None)
    display(df_horario)
    
    # ---------------------------------------------------------
    # 2. SISTEMA DE SUGERENCIAS Y ALERTAS (Análisis a Futuro)
    # ---------------------------------------------------------
    print("\n" + "="*80)
    print("📊 REPORTE DE GESTIÓN Y SUGERENCIAS DE CONTRATACIÓN")
    print("="*80)
    
    alertas = 0
    for idx, row in df_demanda.iterrows():
        id_c = row['ID_Clase']
        cupos_necesarios = row['Cupos_Estimados']
        secciones_pedidas = int(np.ceil(cupos_necesarios / CAPACIDAD_ESTANDAR))
        secciones_logradas = secciones_abiertas_por_clase.get(id_c, 0)
        
        if secciones_pedidas > secciones_logradas:
            alertas += 1
            clase_nombre = row['Nombre_Clase']
            area_id = df_malla[df_malla['ID_Clase'] == id_c]['ID_Area'].values[0]
            
            user = "Joasro"
            password = "Akriila123." 
            host = "localhost"
            db = "dss_academico_unah"

            engine_temp = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{db}")

            # Obtenemos el nombre del área para la alerta
            
            area_nombre = pd.read_sql(f"SELECT Nombre_Area FROM areas_academicas WHERE ID_Area = {area_id}", engine_temp).values[0][0]
            
            print(f"⚠️ DÉFICIT EN: {clase_nombre}")
            print(f"   - Demanda proyectada: {cupos_necesarios} alumnos ({secciones_pedidas} secciones).")
            print(f"   - Cobertura lograda:  {secciones_logradas} secciones abiertas.")
            print(f"   - 💡 SUGERENCIA DIRECTIVA: Para el próximo periodo, se requiere contratar/habilitar")
            print(f"        presupuesto para un Ingeniero con especialidad en el Área: '{area_nombre}'.")
            print("-" * 60)
            
    if alertas == 0:
        print("✅ Excelente: La oferta académica actual cubre el 100% de la demanda proyectada sin déficit.")
        print(f"   Total de secciones programadas: {len(df_horario)}")
        
    pd.reset_option('display.max_rows')

🎓 OFERTA ACADÉMICA GENERADA (Optimizada para el próximo periodo):


,Hora,Clase,Seccion,Modalidad,Docente,Tipo_Contrato,Espacio
8,10:00,Diseño de Compiladores,Sec-1,Virtual,Ing. Oscar Guillermo Hernández,Base,Campus Virtual
13,10:00,Industria del Software,Sec-1,Presencial,Ing. Edis Francisco Romero,Base,Aula 323
4,10:00,Seguridad Informática,Sec-1,Presencial,Ing. Elias Emilio Flores,Base,Aula 111
10,11:00,Inteligencia Artificial,Sec-1,Presencial,Ing. Asalia Alejandra Zavala,Base,Aula 111
2,11:00,Seminario de Investigación,Sec-3,Presencial,Ing. Oscar Guillermo Hernández,Base,Aula 323
19,12:00,Arquitectura de Computadoras,Sec-1,Presencial,Ing. Edis Francisco Romero,Base,Aula 111
15,12:00,Economía Digital,Sec-1,Virtual,Ing. Asalia Alejandra Zavala,Base,Campus Virtual
18,12:00,Liderazgo para el Cambio,Sec-1,Virtual,Ing. Elias Emilio Flores,Base,Campus Virtual
21,12:00,Redes de Datos I,Sec-1,Virtual,Ing. René Velasquez,Tegucigalpa,Campus Virtual
0,12:00,Seminario de Investigación,Sec-1,Presencial,Ing. Oscar Guillermo Hernández,Base,Aula 323



📊 REPORTE DE GESTIÓN Y SUGERENCIAS DE CONTRATACIÓN
⚠️ DÉFICIT EN: Sistemas Operativos II
   - Demanda proyectada: 19 alumnos (1 secciones).
   - Cobertura lograda:  0 secciones abiertas.
   - 💡 SUGERENCIA DIRECTIVA: Para el próximo periodo, se requiere contratar/habilitar
        presupuesto para un Ingeniero con especialidad en el Área: 'Sistemas Operativos'.
------------------------------------------------------------
⚠️ DÉFICIT EN: Diseño Digital
   - Demanda proyectada: 19 alumnos (1 secciones).
   - Cobertura lograda:  0 secciones abiertas.
   - 💡 SUGERENCIA DIRECTIVA: Para el próximo periodo, se requiere contratar/habilitar
        presupuesto para un Ingeniero con especialidad en el Área: 'Arquitectura y Hardware'.
------------------------------------------------------------
⚠️ DÉFICIT EN: Base de Datos II
   - Demanda proyectada: 14 alumnos (1 secciones).
   - Cobertura lograda:  0 secciones abiertas.
   - 💡 SUGERENCIA DIRECTIVA: Para el próximo periodo, se requiere contratar/